# SVM Sentiment Analysis — Amazon Reviews

Analiza sentymentu recenzji Amazon za pomocą SVM z pełnym pipeline'em:
- **TF-IDF** jako reprezentacja tekstu
- **LinearSVC** jako klasyfikator
- **Podział danych**: Train / Validation / Test
- **Cross-validation** na zbiorze treningowym
- **GridSearchCV** na zbiorze train+validation
- **Finalna ewaluacja** na zbiorze testowym

## 1. Importy

In [ ]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import seaborn as sns
import matplotlib.pyplot as plt

print("Importy załadowane pomyślnie.")

## 2. Wczytanie danych

In [ ]:
df = pd.read_csv('data/processed_reviews.csv')

print(f"Liczba próbek: {len(df)}")
print(f"Kolumny: {df.columns.tolist()}")
print("\nRozkład klas (label):")
print(df['label'].value_counts().sort_index())

df.head(3)

## 3. Podział danych: Train / Validation / Test

Strategia podziału:
- **Test set (20%)** — odkładamy na sam koniec, dotykamy go tylko raz
- **Validation set (15% całości = ~19% pozostałych)** — służy do oceny modeli z GridSearch i wyboru najlepszego
- **Train set (65% całości)** — na nim odbywa się cross-validation i trening

```
Cały dataset
│
├── Test (20%)          ← finalna ewaluacja, dotykamy tylko raz na końcu
│
└── Reszta (80%)
    ├── Validation (15% całości ≈ 19% reszty)  ← wybór modelu po GridSearch
    └── Train (65% całości ≈ 81% reszty)        ← CV + GridSearchCV
```

In [ ]:
X = df['full_text']
y = df['label']

# Krok 1: Odłóż zbiór testowy (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Krok 2: Z pozostałych 80% wydziel zbiór walidacyjny (~19% z 80% = 15% całości)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.1875,  # 0.1875 * 0.80 ≈ 0.15 całości
    random_state=42,
    stratify=y_temp
)

print(f"Train:      {len(X_train):>6} próbek ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validation: {len(X_val):>6} próbek ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test:       {len(X_test):>6} próbek ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nRozkład klas w Train:")
print(y_train.value_counts().sort_index())
print(f"\nRozkład klas w Validation:")
print(y_val.value_counts().sort_index())
print(f"\nRozkład klas w Test:")
print(y_test.value_counts().sort_index())

## 4. Baseline Pipeline

Budujemy bazowy pipeline z parametrami wynikającymi z wcześniejszych eksperymentów:
- `stop_words=None` — negacje ('not', 'no') niosą istotną informację o sentymencie
- `ngram_range=(1,3)` — trigramy pomagają łapać frazy jak 'not good at all'
- `C=1` — dobry punkt startowy dla LinearSVC

In [ ]:
baseline_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=30000,
        stop_words=None,      # zachowujemy negacje
        ngram_range=(1, 3),   # uni + bi + trigramy
        lowercase=True,
        min_df=3
    )),
    ('svm', LinearSVC(
        C=1,
        max_iter=3000,
        dual='auto',
        random_state=42
    ))
])

print("Pipeline zdefiniowany:")
print(baseline_pipe)

## 5. Cross-Validation na zbiorze treningowym

CV działa wyłącznie na `X_train` — pipeline gwarantuje brak data leakage
(TF-IDF jest fitowany osobno dla każdego foldu).

In [ ]:
print("Uruchamianie 5-fold Cross-Validation na zbiorze treningowym...")
start = time.time()

cv_scores = cross_val_score(
    baseline_pipe,
    X_train, y_train,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

print(f"Czas: {time.time() - start:.1f}s")
print(f"\nCV F1-macro: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Wyniki per fold: {np.round(cv_scores, 4)}")

# Wizualizacja wyników CV
fig, ax = plt.subplots(figsize=(7, 4))
folds = [f'Fold {i+1}' for i in range(len(cv_scores))]
bars = ax.bar(folds, cv_scores, color='steelblue', alpha=0.8, edgecolor='black')
ax.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'Średnia: {cv_scores.mean():.4f}')
ax.set_ylim(max(0, cv_scores.min() - 0.02), min(1, cv_scores.max() + 0.02))
ax.set_ylabel('F1-macro')
ax.set_title('5-Fold Cross-Validation — Baseline Pipeline')
ax.legend()
for bar, score in zip(bars, cv_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{score:.4f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## 6. Grid Search — dobór optymalnych hiperparametrów

GridSearchCV przeszukuje siatkę parametrów używając wewnętrznej CV na `X_train`.
Siatka jest celowo ograniczona — zbyt duża powoduje kombinatoryczną eksplozję.

**Strategia**: testujemy kluczowe osie zmienności:
- `max_features` i `ngram_range` w TF-IDF
- `C` w LinearSVC (regularyzacja)
- `class_weight` (pomocne przy niezbalansowanych klasach)

In [ ]:
# Pipeline dla GridSearch — osobna instancja, żeby nie modyfikować baseline
gs_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        stop_words=None,
        dual='auto'
    )),
    ('svm', LinearSVC(
        max_iter=3000,
        random_state=42,
        dual='auto'
    ))
])

# Siatka parametrów
# Łącznie: 3 * 2 * 2 * 2 = 24 kombinacje × 3 foldy = 72 fity
param_grid = {
    'tfidf__max_features': [15000, 30000, None],
    'tfidf__ngram_range':  [(1, 2), (1, 3)],
    'svm__C':              [0.3, 1, 5],
    'svm__class_weight':   [None, 'balanced'],
}

grid_search = GridSearchCV(
    gs_pipe,
    param_grid,
    cv=3,               # 3-fold wewnątrz GridSearch (szybciej niż 5)
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    refit=True          # automatycznie refituje best model na całym X_train
)

n_combinations = (3 * 2 * 3 * 2)
print(f"Liczba kombinacji: {n_combinations} × 3 foldy = {n_combinations * 3} fitów")
print("Uruchamianie GridSearchCV...")
start = time.time()

grid_search.fit(X_train, y_train)

print(f"\nGotowe! Czas: {time.time() - start:.1f}s")
print(f"\nNajlepsze parametry:")
for k, v in grid_search.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nNajlepszy CV F1-macro (train): {grid_search.best_score_:.4f}")

In [ ]:
# Przegląd wyników Grid Search — top 10 kombinacji
cv_results = pd.DataFrame(grid_search.cv_results_)
top10 = (cv_results
         .sort_values('mean_test_score', ascending=False)
         .head(10)[['param_tfidf__max_features', 'param_tfidf__ngram_range',
                    'param_svm__C', 'param_svm__class_weight',
                    'mean_test_score', 'std_test_score']]
         .reset_index(drop=True))

top10.columns = ['max_features', 'ngram_range', 'C', 'class_weight',
                 'mean_F1', 'std_F1']
top10['mean_F1'] = top10['mean_F1'].round(4)
top10['std_F1']  = top10['std_F1'].round(4)

print("Top 10 kombinacji hiperparametrów:")
display(top10)

## 7. Ewaluacja na zbiorze walidacyjnym

Zbiór walidacyjny był **niewidoczny** podczas Grid Search.
Używamy go teraz do weryfikacji, czy najlepszy model generalizuje.

In [ ]:
best_model = grid_search.best_estimator_

# Predykcja na zbiorze walidacyjnym
y_val_pred = best_model.predict(X_val)
val_f1 = f1_score(y_val, y_val_pred, average='macro')

print("=" * 55)
print("WYNIKI NA ZBIORZE WALIDACYJNYM (model z GridSearch)")
print("=" * 55)
print(f"CV F1-macro (train, wewnętrzny): {grid_search.best_score_:.4f}")
print(f"F1-macro   (validation):         {val_f1:.4f}")
print()
print(classification_report(
    y_val, y_val_pred,
    target_names=['Negative (1-2)', 'Neutral (3)', 'Positive (4-5)']
))

In [ ]:
# Confusion matrix — zbiór walidacyjny
cm_val = confusion_matrix(y_val, y_val_pred)
cm_val_pct = cm_val.astype(float) / cm_val.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(7, 5))
sns.heatmap(cm_val_pct, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=['Neg (1-2)', 'Neu (3)', 'Pos (4-5)'],
            yticklabels=['Neg (1-2)', 'Neu (3)', 'Pos (4-5)'])
plt.ylabel('Etykieta rzeczywista')
plt.xlabel('Predykcja modelu')
plt.title(f'Confusion Matrix — Zbiór Walidacyjny (F1={val_f1:.4f})')
plt.tight_layout()
plt.show()

## 8. Finalna ewaluacja na zbiorze testowym

> ⚠️ **Uwaga**: Ta komórka jest uruchamiana **tylko raz**, po zakończeniu wszystkich eksperymentów.
> Wyniki testowe nie mogą wpływać na dalsze decyzje modelarskie.

Przed finalnym testem refitujemy model na połączonych danych train + validation,
żeby w pełni wykorzystać wszystkie dostępne oznaczone dane.

In [ ]:
import copy

# Łączymy train + validation → trenujemy final model
X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

# Kopiujemy best pipeline i trenujemy na train+val
final_model = copy.deepcopy(best_model)
final_model.fit(X_trainval, y_trainval)

# Predykcja na zbiorze testowym
y_test_pred = final_model.predict(X_test)
test_f1 = f1_score(y_test, y_test_pred, average='macro')

print("=" * 55)
print("FINALNE WYNIKI NA ZBIORZE TESTOWYM")
print("=" * 55)
print(f"F1-macro (test): {test_f1:.4f}")
print()
print(classification_report(
    y_test, y_test_pred,
    target_names=['Negative (1-2)', 'Neutral (3)', 'Positive (4-5)']
))

In [ ]:
# Confusion matrix — zbiór testowy
cm_test = confusion_matrix(y_test, y_test_pred)
cm_test_pct = cm_test.astype(float) / cm_test.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(7, 5))
sns.heatmap(cm_test_pct, annot=True, fmt='.2%', cmap='Greens',
            xticklabels=['Neg (1-2)', 'Neu (3)', 'Pos (4-5)'],
            yticklabels=['Neg (1-2)', 'Neu (3)', 'Pos (4-5)'])
plt.ylabel('Etykieta rzeczywista')
plt.xlabel('Predykcja modelu')
plt.title(f'Confusion Matrix — Zbiór Testowy (F1={test_f1:.4f})')
plt.tight_layout()
plt.show()

## 9. Podsumowanie wyników

In [ ]:
print("=" * 55)
print("PODSUMOWANIE EKSPERYMENTU")
print("=" * 55)
print(f"\nPodział danych:")
print(f"  Train:      {len(X_train):>6} próbek")
print(f"  Validation: {len(X_val):>6} próbek")
print(f"  Test:       {len(X_test):>6} próbek")

print(f"\nWyniki:")
print(f"  CV F1-macro  (train, 5-fold baseline): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"  CV F1-macro  (train, GridSearch best): {grid_search.best_score_:.4f}")
print(f"  F1-macro     (validation):             {val_f1:.4f}")
print(f"  F1-macro     (test — FINALNA):         {test_f1:.4f}")

print(f"\nNajlepsze hiperparametry (GridSearch):")
for k, v in grid_search.best_params_.items():
    print(f"  {k}: {v}")

## 10. Analiza błędów

In [ ]:
results_df = pd.DataFrame({
    'text':      X_test.values,
    'actual':    y_test.values,
    'predicted': y_test_pred
})

errors = results_df[results_df['actual'] != results_df['predicted']]
print(f"Błędnych predykcji: {len(errors)} / {len(results_df)} "
      f"({len(errors)/len(results_df)*100:.1f}%)")

# Skrajne pomyłki: Negatyw → Pozytyw
label_values = sorted(y.unique())
neg_label = label_values[0]   # najniższa klasa = negatywna
pos_label = label_values[-1]  # najwyższa klasa = pozytywna

extreme = errors[(errors['actual'] == neg_label) & (errors['predicted'] == pos_label)]
print(f"\nSkrajne pomyłki (Negatyw → Pozytyw): {len(extreme)}")

for _, row in extreme.head(5).iterrows():
    print(f"\n[actual={row['actual']} | pred={row['predicted']}]")
    print(f"  {str(row['text'])[:250]}...")